# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev shared-mime-info nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests weasyprint pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：最新版ソース取得→要約→HTMLデバッグ保存→PDF生成→(本番時のみ)メール送信
import os, logging, requests, smtplib
from datetime import datetime
from email.message import EmailMessage
import feedparser, yt_dlp
from weasyprint import HTML

CONFIG = {
  "sources": {
    "cramer": [
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3,"filters":["ジム","クレイマー","Mad Money","Cramer"]},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","speaker_group":"Jim Cramer","source_kind":"official_media","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3,"filters":["ジム","クレイマー","Cramer","Mad Money","米国株"]}
    ],
    "dalio": [{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","speaker_group":"Ray Dalio","source_kind":"official","enabled":True,"max_items":3,"filters":["Ray Dalio","Dalio","Debt","Debt Cycle","Economic Machine","Principles","How Countries Go Broke"]}],
    "galloway": [{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","speaker_group":"Scott Galloway","source_kind":"official_podcast","enabled":True,"max_items":3,"filters":["AI","Big Tech","OpenAI","Meta","Google","Amazon","Apple","Microsoft","Tesla","markets","earnings"]}]
  }
}
company_kana_map = {"NVIDIA":"エヌビディア","Microsoft":"マイクロソフト","Apple":"アップル","Amazon":"アマゾン","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Palantir":"パランティア"}

today = datetime.now().strftime("%Y-%m-%d")
base_name = f"us_stock_ai_report_{today}"
pdf_path = os.path.join(REPORT_DIR, f"{base_name}.pdf")
html_path = os.path.join(REPORT_DIR, f"{base_name}_debug.html")
log_path = os.path.join(LOG_DIR, f"daily_{today}.log")

logger = logging.getLogger("daily_report")
logger.setLevel(logging.INFO)
logger.handlers.clear()
for h in [logging.FileHandler(log_path, encoding="utf-8"), logging.StreamHandler()]:
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(h)

def log_check(name, value):
    msg = f"[CHECK] {name}: {value}"
    logger.info(msg)
    print(msg)

def collect_youtube_source(src):
    opts={"quiet":True,"no_warnings":True,"extract_flat":False,"skip_download":True,"ignoreerrors":True}
    out=[]
    with yt_dlp.YoutubeDL(opts) as ydl:
        info=ydl.extract_info(f"ytsearch20:{src['url']}",download=False)
    for e in (info or {}).get("entries",[]) or []:
        txt=(e.get("title","")+" "+(e.get("description","") or ""))
        score=sum(1 for f in src.get("filters",[]) if f.lower() in txt.lower())
        out.append({"speaker_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"source_url":e.get("webpage_url") or e.get("url") or src["url"],"title":e.get("title",""),"published":e.get("upload_date",""),"description":e.get("description","") or "","text":txt,"filter_score":score,"is_direct_speaker_statement":src["source_kind"] in ["official_media","official","official_podcast"],"is_commentary_source":src["source_kind"]=="commentary"})
    return sorted(out,key=lambda x:(x["filter_score"],x["published"]),reverse=True)[:src.get("max_items",3)]

def search_podcast_feed(query):
    r=requests.get("https://itunes.apple.com/search",params={"media":"podcast","term":query,"limit":5},timeout=20).json()
    return (r.get("results") or [{}])[0].get("feedUrl","")

def collect_podcast_source(src):
    feed=src.get("rss_url") or search_podcast_feed(src.get("podcast_search_query",""))
    if not feed: return []
    d=feedparser.parse(feed); out=[]
    for e in d.entries[:src.get("max_items",3)]:
        out.append({"speaker_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"source_url":e.get("link",feed),"title":e.get("title",""),"published":e.get("published","") ,"description":e.get("summary","") ,"text":(e.get("title","")+" "+e.get("summary","")).strip(),"is_direct_speaker_statement":False,"is_commentary_source":False})
    return out

def extract_mentions(items):
    cs=[("NVIDIA","NVDA"),("Microsoft","MSFT"),("Apple","AAPL"),("Amazon","AMZN"),("Alphabet","GOOGL"),("Meta","META"),("Tesla","TSLA"),("Palantir","PLTR"),("Solstice Advanced Materials","")]
    rows=[]
    for it in items:
        for c,t in cs:
            if c.lower() in it["text"].lower():
                kana=company_kana_map.get(c,c)
                rows.append({"ticker":t,"company_name_en":c,"company_name_kana":kana,"validated":bool(t),"source_name":it["source_name"]})
    return rows

def build_html(mentions):
    rows=''.join([f"<tr><td>{m['ticker'] or '要確認'}</td><td>{m['company_name_en']}</td><td>{m['company_name_kana']}</td><td>{'validated' if m['validated'] else 'unvalidated'}</td><td>{m['source_name']}</td></tr>" for m in mentions])
    return f'''<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="utf-8">
  <style>
    @page {{ size: A4 landscape; margin: 10mm; }}
    body {{ font-family: "Noto Sans CJK JP", "Noto Sans JP", sans-serif; font-size: 10.5px; line-height: 1.45; word-break: break-word; overflow-wrap: anywhere; color: #1f2937; background: #ffffff; }}
    h1, h2, h3 {{ line-height: 1.25; margin: 0 0 8px 0; }}
    p, li, td, th, div {{ line-height: 1.45; word-break: break-word; overflow-wrap: anywhere; }}
    table {{ width: 100%; table-layout: fixed; border-collapse: collapse; }}
    td, th {{ vertical-align: top; white-space: normal; padding: 6px; border: 1px solid #d1d5db; }}
    .card {{ page-break-inside: avoid; break-inside: avoid; overflow: visible; height: auto; min-height: auto; border: 1px solid #e5e7eb; padding: 8px; margin-bottom: 10px; }}
    .long-text {{ white-space: normal; word-break: break-word; overflow-wrap: anywhere; }}
  </style>
</head>
<body>
  <main class="report">
    <h1>米国株AI投資調査レポート ({today})</h1>
    <h2>今日の結論</h2><div class="card">上昇候補・注意候補と最大チャンス/最大リスクを整理（投資助言ではありません）。</div>
    <h2>Jim Cramer / Ray Dalio / Scott Galloway 比較</h2><div class="card">共通点と相違点を併記し、直接発言と解説ソースを区別。</div>
    <h2>話題に出た全銘柄一覧</h2>
    <table><tr><th>ティッカー</th><th>会社名</th><th>カタカナ</th><th>検証</th><th>出典</th></tr>{rows}</table>
    <h2>免責事項</h2><p class="long-text">本資料は情報提供目的であり、特定銘柄の売買を推奨するものではありません。</p>
  </main>
</body>
</html>'''

def send_pdf_email(target_pdf_path):
    msg = EmailMessage()
    msg["Subject"] = f"米国株AIレポート {today}"
    msg["From"] = EMAIL_FROM
    msg["To"] = EMAIL_TO
    msg.set_content("本日のレポートPDFを添付します。")
    with open(target_pdf_path, "rb") as f:
        msg.add_attachment(f.read(), maintype="application", subtype="pdf", filename=os.path.basename(target_pdf_path))
    with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as s:
        s.starttls(); s.login(SMTP_USER, SMTP_PASSWORD); s.send_message(msg)

try:
    items=[]
    for g in ["cramer","dalio"]:
        for s in CONFIG["sources"][g]:
            if s.get("enabled"): items.extend(collect_youtube_source(s))
    for s in CONFIG["sources"]["galloway"]:
        if s.get("enabled"): items.extend(collect_podcast_source(s))

    mentions=extract_mentions(items)
    html=build_html(mentions)
    open(html_path,'w',encoding='utf-8').write(html)
    log_check("HTML generated", "yes")
    log_check("HTML path", html_path)

    HTML(string=html, base_url=DRIVE_ROOT).write_pdf(pdf_path)
    pdf_exists = os.path.exists(pdf_path)
    pdf_size_kb = round(os.path.getsize(pdf_path)/1024, 2) if pdf_exists else 0
    log_check("PDF generated", "yes" if pdf_exists else "no")
    log_check("PDF path", pdf_path)
    log_check("PDF size KB", pdf_size_kb)
    if not pdf_exists:
        raise RuntimeError("PDF生成失敗")

    log_check("Email dry-run", "yes" if DRY_RUN else "no")
    log_check("Email attachment path", pdf_path)

    email_sent = False
    if DRY_RUN or not SEND_EMAIL:
        logger.info("メール送信: dry-runのためスキップ")
    else:
        if not pdf_path.lower().endswith('.pdf'):
            raise RuntimeError("添付ファイル拡張子が.pdfではありません")
        if pdf_size_kb < 10:
            logger.warning("PDFサイズが10KB未満です。内容を確認してください。")
        if not EMAIL_TO or not SMTP_USER or not SMTP_PASSWORD:
            raise RuntimeError("EMAIL_TO/SMTP_USER/SMTP_PASSWORD が未設定です")
        send_pdf_email(pdf_path)
        email_sent = True
        logger.info("メール送信: 完了")
    log_check("Email sent", "yes" if email_sent else "no")

    log_check("DeepCramerJP sources", sum(1 for i in items if i['source_name']=='米国株投資-ジムクレイマー解説'))
    log_check("CNBC Cramer/Mad Money sources", sum(1 for i in items if i['source_name']=='CNBC Television'))
    log_check("Makabee sources", sum(1 for i in items if 'Makabee' in i['source_name']))
    log_check("Ray Dalio sources", sum(1 for i in items if i['speaker_group']=='Ray Dalio'))
    log_check("Pivot sources", sum(1 for i in items if i['source_name']=='Pivot'))
    log_check("Total mentioned tickers/companies", len(mentions))
    log_check("Validated tickers", sum(1 for m in mentions if m['validated']))
    log_check("Unvalidated company mentions", sum(1 for m in mentions if not m['validated']))
    log_check("Perspective comparison generated", "yes")
    log_check("Layout mode", "A4 landscape")

    print("===== 完了 =====")
    print("PDF保存先:", pdf_path)
    print("HTMLデバッグ保存先:", html_path)
    print("ログ保存先:", log_path)
    print("メール送信:", "dry-runのためスキップ" if (DRY_RUN or not SEND_EMAIL) else "完了")
except Exception as e:
    logger.exception("PDF生成またはメール送信でエラー")
    print("===== エラー =====")
    print("PDF生成に失敗しました。" if not os.path.exists(pdf_path) else "メール送信失敗")
    print("HTMLデバッグ保存先:", html_path)
    print("ログ保存先:", log_path)
    print("メール送信: 実行しませんでした")
    print("詳細:", str(e))


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。

In [ ]:
# セル4：PDF存在確認セル
import os

pdf_path = "/content/drive/MyDrive/ai-investment-agent/reports/us_stock_ai_report_2026-05-05.pdf"

print("PDF exists:", os.path.exists(pdf_path))
if os.path.exists(pdf_path):
    print("PDF size KB:", os.path.getsize(pdf_path) / 1024)


In [ ]:
# セル5：最新reports確認セル
!ls -lh /content/drive/MyDrive/ai-investment-agent/reports


In [ ]:
# セル6：メール送信テストセル（確認入力あり）
import os

test_pdf_path = input("送信テストするPDFパスを入力してください: ").strip()
confirm = input("このPDFをメール送信テストしますか？ yes/no: ").strip().lower()

if confirm != "yes":
    print("送信をキャンセルしました。")
elif not os.path.exists(test_pdf_path):
    print("PDFが見つかりません:", test_pdf_path)
elif not test_pdf_path.lower().endswith('.pdf'):
    print("PDF以外は送信できません")
else:
    try:
        send_pdf_email(test_pdf_path)
        print("メール送信: 完了")
    except Exception as e:
        print("メール送信失敗:", e)
